# Notebook 13 — Does the v8 foundation actually use cross-instrument signal?

**Question.** The v8 MAE objective masks one of 10 bands and reconstructs it from the other 9. Nothing in the loss penalises the encoder for ignoring Euclid bands when reconstructing a Rubin band — Rubin r and i alone could be enough. So the cross-instrument flow in the bottleneck might be weak even though the training setup *exposes* the model to cross-instrument prediction.

**The wavelength-distance confound.** A naive ablation ("zero out Euclid, see if Rubin reconstruction degrades") cannot distinguish two scenarios:

1. The model genuinely couples instruments and degrades because cross-instrument info was useful.
2. The model ignores Euclid; degradation is just because Rubin r/i are spectrally adjacent, Euclid bands aren't.

Both look the same in a simple ablation.

**Two cheap diagnostics that do better.**

- **Diagnostic 1 — Input-pixel gradient attribution.** Backprop the reconstruction loss to the input pixels of every band and measure per-band gradient magnitude. This directly answers "what is the model actually using", regardless of what is *intrinsically informative*. A model that has learned to ignore Euclid will produce essentially zero gradient on Euclid inputs.
- **Diagnostic 2 — Probe on the bottleneck.** Train regressors `bottleneck → flux in band X` for each of the 10 bands. Run two flavours: (a) random-point sampling + ridge (the simple sanity check) and (b) **source-centered sampling + small MLP** (the honest probe of "is the info present at all", since the foundation's per-target decoders also use nonlinear capacity downstream of the bottleneck). If Euclid bands are well-recovered, the encoder is preserving Euclid information. If they aren't, the encoder is dropping it.

Together these tell us whether v9 adversarial masking (mask Rubin r+g+i jointly so the model can't shortcut) is needed.

**Status:** This notebook is a self-contained diagnostic. It does not retrain anything. Runs on a single GPU using existing tile data and the v8 foundation checkpoint.

## Decision rule

| Diagnostic 1 (attribution) | Diagnostic 2 (linear probe) | Verdict |
|---|---|---|
| Cross-instrument attribution > 30 % of within-instrument | MLP-probe Euclid R² ≈ Rubin R² | Cross-instrument signal is in the bottleneck — v9 adversarial masking not urgent. |
| Cross-instrument > 30 % | MLP-probe Euclid R² ≪ Rubin R² | Encoder uses Euclid (likely VIS) as a spatial scaffold, but the bottleneck stores Rubin SED content. Worth a foundation upgrade. |
| Cross-instrument ≪ within | Either | Encoder ignores cross-instrument inputs. v9 adversarial masking justified. |


## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

ROOT = Path('/home/shemmati/Work/Projects/JAISP').resolve()
MODELS = ROOT / 'models'
if str(MODELS) not in sys.path:
    sys.path.insert(0, str(MODELS))

from load_foundation import load_foundation
from jaisp_foundation_v7 import (
    ALL_BANDS, RUBIN_BANDS, EUCLID_BANDS, band_group, STREAM_PIXEL_SCALES,
)
from jaisp_dataset_v6 import JAISPDatasetV6

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Bands:', ALL_BANDS)

# Output directory for any figures we save.
OUT = ROOT / 'io' / '_nb13_outputs'
OUT.mkdir(parents=True, exist_ok=True)
FIGS = ROOT / 'docs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load v8 foundation (frozen at inference).
CKPT = ROOT / 'models' / 'checkpoints' / 'jaisp_v8_fine' / 'checkpoint_best.pt'
model = load_foundation(str(CKPT), device=DEVICE, freeze=False)
model.eval()
for p in model.parameters():
    p.requires_grad = False  # freeze everything; only inputs will require grad in diag 1
n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded v8 foundation, {n_params/1e6:.1f}M params total')

In [ ]:
# Build a small dataset (no augmentation) and grab a handful of paired tiles.
RUBIN_DIR = ROOT / 'data' / 'rubin_tiles_all'
EUCLID_DIR = ROOT / 'data' / 'euclid_tiles_all'

dataset = JAISPDatasetV6(
    rubin_dir=str(RUBIN_DIR),
    euclid_dir=str(EUCLID_DIR),
    load_euclid=True,
    augment=False,
)
print(f'Dataset has {len(dataset)} tiles total.')

# Pick the first N tiles that have all 10 bands available.
N_TILES = 6  # bump this for tighter statistics; 6 is enough for a smoke test.
selected = []
for idx in range(len(dataset)):
    if len(selected) >= N_TILES:
        break
    sample = dataset[idx]
    if not sample['has_euclid']:
        continue
    have_bands = set(sample['rubin'].keys()) | set(sample['euclid'].keys())
    if all(b in have_bands for b in ALL_BANDS):
        selected.append(idx)
print(f'Using {len(selected)} fully-paired tiles: {selected}')

In [ ]:
def sample_to_band_dicts(sample, device=DEVICE):
    """Pack a JAISPDatasetV6 sample into the {band: tensor[1, 1, H, W]} dicts the model expects.

    Returns four dicts: context_images, context_rms, target_image_per_band,
    target_rms_per_band.  At call time we'll choose which band to mask;
    everything else stays as context."""
    images = {}
    rms = {}
    for b, d in sample['rubin'].items():
        images[b] = d['image'].to(device).unsqueeze(0).float()  # [1, 1, H, W]
        rms[b]    = d['rms'].to(device).unsqueeze(0).float()
    for b, d in sample['euclid'].items():
        images[b] = d['image'].to(device).unsqueeze(0).float()
        rms[b]    = d['rms'].to(device).unsqueeze(0).float()
    return images, rms

imgs, rmsd = sample_to_band_dicts(dataset[selected[0]])
for b in ALL_BANDS:
    print(f'  {b:14s}  image {tuple(imgs[b].shape)}  rms {tuple(rmsd[b].shape)}')

## 1. Diagnostic 1 — Input gradient attribution

**What this measures.** For each pair `(target_band, source_band)`, take the foundation's reconstruction loss when `target_band` is masked, then backprop to the input pixels of `source_band` and record the mean absolute gradient. This is a direct measure of how strongly the encoder uses `source_band` to predict `target_band` — independent of what is intrinsically informative.

**What it does not measure.** The magnitude of the gradient depends on input scale, so we normalise by the input's per-pixel RMS to make bands comparable.

**Robustness to the wavelength-distance confound.** If the model has *learned* that Euclid is redundant given Rubin r/i, the gradient on Euclid pixels will be small because the encoder isn't using them — even though Euclid pixels are *intrinsically* informative about a Rubin g pixel through galaxy SEDs. That is exactly the failure mode we want to detect.

In [ ]:
def attribution_for_target(model, sample, target_band, device=DEVICE):
    """Return a dict {source_band: scalar attribution} for the given target_band.

    Attribution = mean(|grad of loss w.r.t. source-band input pixels|) / mean(|input|).
    The denominator removes scale differences across bands (Rubin and Euclid have
    very different flux units after RMS normalisation)."""
    imgs, rmsd = sample_to_band_dicts(sample, device=device)
    target_image = imgs[target_band]
    target_rms = rmsd[target_band]

    # Build context = all bands except the target. Mark each context input as a leaf with grad.
    context_imgs = {}
    context_rms = {}
    for b in ALL_BANDS:
        if b == target_band:
            continue
        x = imgs[b].clone().detach().requires_grad_(True)
        context_imgs[b] = x
        context_rms[b] = rmsd[b]

    out = model(
        context_images=context_imgs,
        context_rms=context_rms,
        target_band=target_band,
        target_image=target_image,
        target_rms=target_rms,
    )
    loss = out['loss']
    loss.backward()

    attr = {}
    for b, x in context_imgs.items():
        if x.grad is None:
            attr[b] = 0.0
        else:
            g = x.grad.detach().abs().mean().item()
            scale = x.detach().abs().mean().item() + 1e-8
            attr[b] = g / scale
    return attr, float(loss.item())

In [ ]:
# Build the 10x10 attribution matrix (target row, source column).
n_b = len(ALL_BANDS)
M = np.zeros((n_b, n_b), dtype=np.float64)        # accumulator
loss_per_target = np.zeros(n_b, dtype=np.float64)  # mean recon loss per target
counts = np.zeros(n_b, dtype=np.int64)

for tile_i, idx in enumerate(selected):
    sample = dataset[idx]
    print(f'\nTile {tile_i+1}/{len(selected)} (id={sample["tile_id"]})')
    for ti, target in enumerate(ALL_BANDS):
        attr, l = attribution_for_target(model, sample, target)
        for sj, src in enumerate(ALL_BANDS):
            if src == target:
                continue
            M[ti, sj] += attr.get(src, 0.0)
        loss_per_target[ti] += l
        counts[ti] += 1
        print(f'  target={target:14s}  loss={l:.4f}', end='\r')
    print('')
M /= np.maximum(counts[:, None], 1)
loss_per_target /= np.maximum(counts, 1)
print('\nAttribution matrix shape:', M.shape)
print('Mean reconstruction loss per target band:')
for ti, b in enumerate(ALL_BANDS):
    print(f'  {b:14s}  loss={loss_per_target[ti]:.4f}')

In [ ]:
# Row-normalise: each row sums to 1, making the row a probability distribution
# over source bands.  The diagonal (self-attribution) is masked since the target
# is never in the context.
M_row = M.copy()
row_sums = M_row.sum(axis=1, keepdims=True)
M_row = np.where(row_sums > 0, M_row / np.maximum(row_sums, 1e-30), 0.0)

# Build a band-class map for highlighting cross-instrument cells.
is_rubin = np.array([b in RUBIN_BANDS for b in ALL_BANDS])
is_euclid = ~is_rubin

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={'width_ratios': [1, 1]})

# Left: raw attribution magnitude.
im0 = axes[0].imshow(M, cmap='viridis')
axes[0].set_title('Raw attribution (mean |grad| / mean |input|)')
axes[0].set_xticks(range(n_b)); axes[0].set_yticks(range(n_b))
axes[0].set_xticklabels(ALL_BANDS, rotation=45, ha='right')
axes[0].set_yticklabels(ALL_BANDS)
axes[0].set_ylabel('Target (masked) band')
axes[0].set_xlabel('Source band')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Right: row-normalised — distribution of attribution per target.
im1 = axes[1].imshow(M_row, cmap='magma', vmin=0, vmax=max(0.3, M_row.max()))
axes[1].set_title('Row-normalised: fraction of attribution per source')
axes[1].set_xticks(range(n_b)); axes[1].set_yticks(range(n_b))
axes[1].set_xticklabels(ALL_BANDS, rotation=45, ha='right')
axes[1].set_yticklabels(ALL_BANDS)
axes[1].set_ylabel('Target (masked) band')
axes[1].set_xlabel('Source band')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Draw a divider between Rubin (rows/cols 0-5) and Euclid (rows/cols 6-9).
for ax in axes:
    ax.axhline(5.5, color='white', lw=0.7, ls='--')
    ax.axvline(5.5, color='white', lw=0.7, ls='--')

fig.suptitle(f'v8 foundation — input gradient attribution ({len(selected)} tiles)', y=1.02)
fig.tight_layout()
fig.savefig(FIGS / 'cross_instrument_attribution.png', dpi=160, bbox_inches='tight')
fig.savefig(OUT / 'cross_instrument_attribution.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved:', FIGS / 'cross_instrument_attribution.png')

In [ ]:
# Summary numbers: for each target band, what fraction of attribution lands on
# (a) the same instrument's other bands vs (b) the cross-instrument bands?
print('Per-target fractional attribution (within-instrument | cross-instrument):')
print('-' * 60)
frac_within = np.zeros(n_b)
frac_cross  = np.zeros(n_b)
for ti, b in enumerate(ALL_BANDS):
    same = is_rubin if b in RUBIN_BANDS else is_euclid
    frac_within[ti] = M_row[ti, same].sum()
    frac_cross[ti]  = M_row[ti, ~same].sum()
    print(f'  {b:14s}  within={frac_within[ti]*100:5.1f}%   cross={frac_cross[ti]*100:5.1f}%   '
          f'cross/within={frac_cross[ti]/max(frac_within[ti], 1e-9):.2f}')
print('-' * 60)
mean_cross_over_within = float(np.mean(frac_cross / np.maximum(frac_within, 1e-9)))
print(f'Mean cross/within ratio across all targets: {mean_cross_over_within:.2f}')
print('')
if mean_cross_over_within > 0.3:
    print('Diagnostic 1: cross-instrument signal IS being used (cross/within > 0.30).')
elif mean_cross_over_within > 0.1:
    print('Diagnostic 1: cross-instrument signal is weak (0.10 < cross/within < 0.30).')
else:
    print('Diagnostic 1: cross-instrument signal is essentially absent (cross/within < 0.10).')

## 2. Diagnostic 2 — Linear probe on the bottleneck

**What this measures.** Train a linear ridge regressor `bottleneck → flux in band X` for each of the 10 bands, given a single forward pass that uses *all* 10 bands as input. If the bottleneck encodes Euclid information, the Euclid-band probes should reach a comparable R² to the Rubin-band probes. If the encoder discards Euclid info, the Euclid probes will fail.

**Caveat.** This uses the full bottleneck (no input ablation), so a high Euclid R² doesn't *prove* that the Rubin path contributed to that information — it just shows the bottleneck preserves Euclid info. Combined with diagnostic 1 (which probes input flow) this gives the full picture.

In [ ]:
@torch.no_grad()
def get_bottleneck_and_band_targets(model, sample, n_samples=2000, rng=None, device=DEVICE):
    """Forward all 10 bands → bottleneck. At n_samples random points (in fused-grid
    coordinates), record (bottleneck_feature_256d, log_flux_per_band).

    Per-band log_flux is a 5x5 mean-flux box centred on the corresponding
    pixel in the band's native image, divided by the local RMS (so we're
    measuring local SNR in each band — robust across instruments)."""
    rng = rng or np.random.RandomState(0)
    imgs, rmsd = sample_to_band_dicts(sample, device=device)
    enc = model.encode(context_images=imgs, context_rms=rmsd)
    bottleneck = enc['bottleneck']  # [1, C, H_b, W_b]
    _, C, H_b, W_b = bottleneck.shape
    fused_scale = model.encoder.fused_pixel_scale_arcsec

    # Sample n_samples random fused-grid (y, x) integer coordinates (avoiding edges).
    margin = 4
    ys = rng.randint(margin, H_b - margin, size=n_samples)
    xs = rng.randint(margin, W_b - margin, size=n_samples)

    feats = bottleneck[0, :, ys, xs].T.cpu().numpy()  # [n_samples, C]

    # For each band, compute log10(flux/rms) in a 5x5 box centred on the
    # corresponding native-resolution pixel.
    targets = {}
    for b in ALL_BANDS:
        if b not in imgs:
            continue
        native_scale = STREAM_PIXEL_SCALES[band_group(b)]  # 0.2 (Rubin) or 0.1 (Euclid) arcsec/px
        ratio = fused_scale / native_scale
        ys_n = np.clip((ys * ratio).astype(int), 2, imgs[b].shape[-2] - 3)
        xs_n = np.clip((xs * ratio).astype(int), 2, imgs[b].shape[-1] - 3)
        img_t = imgs[b][0, 0].cpu().numpy()
        rms_t = rmsd[b][0, 0].cpu().numpy()
        vals = np.zeros(n_samples, dtype=np.float32)
        for k in range(n_samples):
            y, x = ys_n[k], xs_n[k]
            patch = img_t[y-2:y+3, x-2:x+3]
            r = rms_t[y-2:y+3, x-2:x+3]
            snr = patch.mean() / (r.mean() + 1e-9)
            vals[k] = np.log10(np.maximum(snr, 1e-3))
        targets[b] = vals

    # Also record (ys, xs) so we can do spatial holdouts.
    return feats, targets, ys, xs

In [ ]:
# Aggregate features and targets across the selected tiles.
rng = np.random.RandomState(42)
all_feats = []
all_targets = {b: [] for b in ALL_BANDS}
all_ys, all_xs, all_tids = [], [], []

N_PER_TILE = 1500
for tile_i, idx in enumerate(selected):
    sample = dataset[idx]
    feats, tgts, ys, xs = get_bottleneck_and_band_targets(
        model, sample, n_samples=N_PER_TILE, rng=rng,
    )
    all_feats.append(feats)
    for b, v in tgts.items():
        all_targets[b].append(v)
    all_ys.append(ys); all_xs.append(xs)
    all_tids.append(np.full(len(ys), tile_i, dtype=np.int32))
    print(f'  tile {tile_i+1}/{len(selected)}  features {feats.shape}')

X = np.concatenate(all_feats, axis=0)
Y = {b: np.concatenate(v, axis=0) for b, v in all_targets.items() if v}
ys_all = np.concatenate(all_ys); xs_all = np.concatenate(all_xs)
tids = np.concatenate(all_tids)

print(f'\nFeature matrix X: {X.shape}')
for b in ALL_BANDS:
    if b in Y:
        print(f'  target {b:14s}  N={len(Y[b])}  mean={Y[b].mean():.2f}  std={Y[b].std():.2f}')

In [ ]:
# Spatial-holdout ridge regression: hold out one tile at a time, train on the rest.
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

r2_per_band = {}
r2_baseline = {}

for b in ALL_BANDS:
    if b not in Y:
        continue
    y = Y[b]
    r2_folds = []
    for held_tile in range(len(selected)):
        train_mask = tids != held_tile
        test_mask  = ~train_mask
        if test_mask.sum() < 50 or train_mask.sum() < 50:
            continue
        reg = Ridge(alpha=1.0)
        reg.fit(X[train_mask], y[train_mask])
        pred = reg.predict(X[test_mask])
        r2_folds.append(r2_score(y[test_mask], pred))
    r2_per_band[b] = float(np.mean(r2_folds)) if r2_folds else float('nan')
    # Baseline: predict the global mean of train.
    base_folds = []
    for held_tile in range(len(selected)):
        train_mask = tids != held_tile
        test_mask  = ~train_mask
        if test_mask.sum() < 50 or train_mask.sum() < 50:
            continue
        pred = np.full(test_mask.sum(), y[train_mask].mean())
        base_folds.append(r2_score(y[test_mask], pred))
    r2_baseline[b] = float(np.mean(base_folds)) if base_folds else float('nan')

print('Linear-probe R² per band (spatial holdout, Ridge alpha=1.0):')
print('-' * 60)
for b in ALL_BANDS:
    if b not in r2_per_band:
        continue
    print(f'  {b:14s}  R²={r2_per_band[b]:+.3f}   baseline={r2_baseline[b]:+.3f}')

rubin_r2  = [r2_per_band[b] for b in RUBIN_BANDS  if b in r2_per_band]
euclid_r2 = [r2_per_band[b] for b in EUCLID_BANDS if b in r2_per_band]
print('-' * 60)
print(f'  Rubin  mean R²  = {np.mean(rubin_r2):.3f}')
print(f'  Euclid mean R²  = {np.mean(euclid_r2):.3f}')
print(f'  Euclid / Rubin  = {np.mean(euclid_r2) / max(np.mean(rubin_r2), 1e-9):.2f}')

In [ ]:
# Bar chart: R² per band, coloured by instrument.
bands_for_plot = [b for b in ALL_BANDS if b in r2_per_band]
vals = np.array([r2_per_band[b] for b in bands_for_plot])
colors = ['#4575b4' if b in RUBIN_BANDS else '#d73027' for b in bands_for_plot]

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(np.arange(len(bands_for_plot)), vals, color=colors)
ax.set_xticks(np.arange(len(bands_for_plot)))
ax.set_xticklabels(bands_for_plot, rotation=45, ha='right')
ax.axhline(0, color='black', lw=0.7)
ax.set_ylabel('R² (spatial holdout)')
ax.set_title('Linear probe: bottleneck → log(SNR per band)\nblue = Rubin, red = Euclid')
ax.grid(axis='y', alpha=0.3, linestyle='--')
fig.tight_layout()
fig.savefig(FIGS / 'cross_instrument_linear_probe.png', dpi=160, bbox_inches='tight')
fig.savefig(OUT / 'cross_instrument_linear_probe.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved:', FIGS / 'cross_instrument_linear_probe.png')

## 2b. Diagnostic 2b — Source-centered sampling + MLP probe

**Why this exists.** Diagnostic 2 (random points, ridge) showed Rubin R² ≈ 0.31 vs Euclid R² ≈ 0.14. Two confounds before drawing conclusions:

1. **Random-point sampling is noisy.** Most points are blank sky; the variance is dominated by rare bright sources. A linear probe at random points measures *image statistics*, not *source content*.
2. **Linear probe = strong assumption.** If the bottleneck stores Euclid info compositionally (extractable only with a small nonlinear head), a ridge probe will report low R² even when the info is there. The Euclid `target_decoder` in the foundation has its own ConvNeXt blocks that operate on the bottleneck — so a 2-layer MLP is the more honest probe of "is the information present at all".

This section repeats the linear probe at **source-centered positions** (peaks detected in VIS) using **both ridge and a small MLP**, so we can disentangle: is the Rubin/Euclid gap from sampling? from probe capacity? or from genuine bottleneck content?


In [ ]:
from scipy.ndimage import gaussian_filter, maximum_filter
import torch.nn as nn

def detect_vis_peaks(vis_img, vis_rms, snr_threshold=8.0, min_dist=12, max_peaks=2000):
    """Find local maxima in VIS at native resolution. Returns (ys, xs) in VIS pixels.
    Uses a smoothed SNR map and a min-distance non-max suppression."""
    snr = vis_img / np.maximum(vis_rms, 1e-9)
    smoothed = gaussian_filter(snr, sigma=1.5)
    local_max = maximum_filter(smoothed, size=min_dist) == smoothed
    above = smoothed > snr_threshold
    valid = vis_rms > 0
    ys, xs = np.where(local_max & above & valid)
    if len(ys) > max_peaks:
        # Keep the brightest.
        order = np.argsort(smoothed[ys, xs])[::-1][:max_peaks]
        ys, xs = ys[order], xs[order]
    return ys, xs

@torch.no_grad()
def get_source_centered_features(model, sample, device=DEVICE):
    """Forward all 10 bands → bottleneck. Detect VIS peaks. Sample bottleneck features
    and per-band native-resolution log SNR at each detected source."""
    imgs, rmsd = sample_to_band_dicts(sample, device=device)
    enc = model.encode(context_images=imgs, context_rms=rmsd)
    bottleneck = enc['bottleneck']  # [1, C, H_b, W_b]
    _, C, H_b, W_b = bottleneck.shape
    fused_scale = model.encoder.fused_pixel_scale_arcsec

    vis_img = imgs['euclid_VIS'][0, 0].cpu().numpy()
    vis_rms = rmsd['euclid_VIS'][0, 0].cpu().numpy()
    ys_vis, xs_vis = detect_vis_peaks(vis_img, vis_rms)

    # VIS pixels (0.1"/px) → bottleneck pixels (0.4"/px): divide by 4.
    bsy = (ys_vis * STREAM_PIXEL_SCALES['euclid'] / fused_scale).astype(int)
    bsx = (xs_vis * STREAM_PIXEL_SCALES['euclid'] / fused_scale).astype(int)
    keep = (bsy >= 4) & (bsy < H_b - 4) & (bsx >= 4) & (bsx < W_b - 4)
    bsy, bsx = bsy[keep], bsx[keep]
    if len(bsy) == 0:
        return np.zeros((0, C), dtype=np.float32), {}, bsy, bsx

    feats = bottleneck[0, :, bsy, bsx].T.cpu().numpy().astype(np.float32)  # [N, C]

    targets = {}
    for b in ALL_BANDS:
        if b not in imgs:
            continue
        native_scale = STREAM_PIXEL_SCALES[band_group(b)]
        ratio = fused_scale / native_scale
        ys_n = np.clip((bsy * ratio).astype(int), 2, imgs[b].shape[-2] - 3)
        xs_n = np.clip((bsx * ratio).astype(int), 2, imgs[b].shape[-1] - 3)
        img_t = imgs[b][0, 0].cpu().numpy()
        rms_t = rmsd[b][0, 0].cpu().numpy()
        vals = np.zeros(len(bsy), dtype=np.float32)
        for k in range(len(bsy)):
            y, x = ys_n[k], xs_n[k]
            patch = img_t[y-2:y+3, x-2:x+3]
            r = rms_t[y-2:y+3, x-2:x+3]
            snr = patch.mean() / (r.mean() + 1e-9)
            vals[k] = np.log10(np.maximum(snr, 1e-3))
        targets[b] = vals
    return feats, targets, bsy, bsx


In [ ]:
# Aggregate source-centered features across the selected tiles.
rng = np.random.RandomState(7)
all_feats_sc = []
all_targets_sc = {b: [] for b in ALL_BANDS}
tids_sc = []

for tile_i, idx in enumerate(selected):
    sample = dataset[idx]
    feats, tgts, bsy, bsx = get_source_centered_features(model, sample)
    if len(feats) == 0:
        print(f'  tile {tile_i+1}/{len(selected)}  no peaks found, skipping')
        continue
    all_feats_sc.append(feats)
    for b, v in tgts.items():
        all_targets_sc[b].append(v)
    tids_sc.append(np.full(len(feats), tile_i, dtype=np.int32))
    print(f'  tile {tile_i+1}/{len(selected)}  source-centered features {feats.shape} ({len(feats)} VIS peaks)')

X_sc = np.concatenate(all_feats_sc, axis=0)
Y_sc = {b: np.concatenate(v, axis=0) for b, v in all_targets_sc.items() if v}
tids_sc_arr = np.concatenate(tids_sc)

print(f'\nSource-centered feature matrix: {X_sc.shape}')
print(f'Total sources: {len(tids_sc_arr)} across {len(np.unique(tids_sc_arr))} tiles')


In [ ]:
# Small MLP probe — same input dimension as Ridge but with a nonlinear hidden layer.
class MLPProbe(nn.Module):
    def __init__(self, in_dim, hidden=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp_probe(X_train, y_train, X_test, y_test, epochs=120, lr=2e-3, batch=256, device=DEVICE):
    """Train MLP and return best test R² over the run."""
    Xt = torch.from_numpy(X_train).float().to(device)
    yt = torch.from_numpy(y_train).float().to(device)
    Xe = torch.from_numpy(X_test).float().to(device)
    ye = torch.from_numpy(y_test).float().to(device)
    probe = MLPProbe(in_dim=X_train.shape[1]).to(device)
    opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_r2 = -float('inf')
    for ep in range(epochs):
        probe.train()
        perm = torch.randperm(len(Xt), device=device)
        for i in range(0, len(Xt), batch):
            sel = perm[i:i+batch]
            pred = probe(Xt[sel])
            loss = ((pred - yt[sel])**2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        probe.eval()
        with torch.no_grad():
            pred = probe(Xe).cpu().numpy()
        ss_res = ((y_test - pred)**2).sum()
        ss_tot = ((y_test - y_test.mean())**2).sum()
        r2 = 1.0 - ss_res / max(ss_tot, 1e-9)
        best_r2 = max(best_r2, r2)
    return float(best_r2)


In [ ]:
# Run BOTH ridge and MLP probes on source-centered features, leave-one-tile-out.
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

r2_ridge_sc = {}
r2_mlp_sc = {}
n_tiles_used = len(np.unique(tids_sc_arr))
print(f'Leave-one-tile-out across {n_tiles_used} tiles, {X_sc.shape[0]} sources, {X_sc.shape[1]}-d features.')

for b in ALL_BANDS:
    if b not in Y_sc:
        continue
    y = Y_sc[b]
    ridge_folds = []
    mlp_folds = []
    for held in np.unique(tids_sc_arr):
        train_mask = tids_sc_arr != held
        test_mask  = ~train_mask
        if test_mask.sum() < 30 or train_mask.sum() < 30:
            continue
        # Ridge
        reg = Ridge(alpha=1.0)
        reg.fit(X_sc[train_mask], y[train_mask])
        ridge_folds.append(r2_score(y[test_mask], reg.predict(X_sc[test_mask])))
        # MLP
        mlp_folds.append(train_mlp_probe(
            X_sc[train_mask], y[train_mask], X_sc[test_mask], y[test_mask],
        ))
    r2_ridge_sc[b] = float(np.mean(ridge_folds)) if ridge_folds else float('nan')
    r2_mlp_sc[b]   = float(np.mean(mlp_folds))   if mlp_folds   else float('nan')
    print(f'  {b:14s}  ridge R²={r2_ridge_sc[b]:+.3f}   MLP R²={r2_mlp_sc[b]:+.3f}   '
          f'(MLP–ridge = {r2_mlp_sc[b] - r2_ridge_sc[b]:+.3f})')


In [ ]:
# Comparison plot: random+ridge (Diag 2) vs source-centered+ridge (Diag 2b ridge)
# vs source-centered+MLP (Diag 2b MLP) per band.
bands_for_plot = [b for b in ALL_BANDS if b in r2_per_band and b in r2_ridge_sc]
v_random_ridge = np.array([r2_per_band[b]   for b in bands_for_plot])
v_source_ridge = np.array([r2_ridge_sc[b]   for b in bands_for_plot])
v_source_mlp   = np.array([r2_mlp_sc[b]     for b in bands_for_plot])

x = np.arange(len(bands_for_plot))
w = 0.27

fig, ax = plt.subplots(figsize=(10, 4.0))
ax.bar(x - w, v_random_ridge, w, label='Random + Ridge (Diag 2)',         color='#9ec5e8')
ax.bar(x,     v_source_ridge, w, label='Source-centered + Ridge (Diag 2b)', color='#4575b4')
ax.bar(x + w, v_source_mlp,   w, label='Source-centered + MLP (Diag 2b)',   color='#0b3d91')
ax.set_xticks(x)
ax.set_xticklabels(bands_for_plot, rotation=45, ha='right')
ax.axhline(0, color='black', lw=0.7)
ax.set_ylabel('R² (spatial holdout)')
ax.set_title('Linear vs nonlinear probe on the bottleneck — random vs source-centered sampling')
ax.grid(axis='y', alpha=0.3, linestyle='--')
# Mark Rubin/Euclid divider
for i, b in enumerate(bands_for_plot):
    if b == 'euclid_VIS':
        ax.axvline(i - 0.5, color='grey', lw=0.6, ls=':')
        break
ax.legend(fontsize=9, loc='upper right')
fig.tight_layout()
fig.savefig(FIGS / 'cross_instrument_linear_probe_v2.png', dpi=160, bbox_inches='tight')
fig.savefig(OUT  / 'cross_instrument_linear_probe_v2.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved:', FIGS / 'cross_instrument_linear_probe_v2.png')


In [ ]:
# Updated headline numbers for the verdict.
rubin_ridge_sc  = float(np.mean([r2_ridge_sc[b] for b in RUBIN_BANDS  if b in r2_ridge_sc]))
euclid_ridge_sc = float(np.mean([r2_ridge_sc[b] for b in EUCLID_BANDS if b in r2_ridge_sc]))
rubin_mlp_sc    = float(np.mean([r2_mlp_sc[b]   for b in RUBIN_BANDS  if b in r2_mlp_sc]))
euclid_mlp_sc   = float(np.mean([r2_mlp_sc[b]   for b in EUCLID_BANDS if b in r2_mlp_sc]))

print('Source-centered probe summary:')
print(f'  Ridge:  Rubin mean R² = {rubin_ridge_sc:.3f}, Euclid mean R² = {euclid_ridge_sc:.3f}, '
      f'ratio = {euclid_ridge_sc/max(rubin_ridge_sc, 1e-9):.2f}')
print(f'  MLP:    Rubin mean R² = {rubin_mlp_sc:.3f}, Euclid mean R² = {euclid_mlp_sc:.3f}, '
      f'ratio = {euclid_mlp_sc/max(rubin_mlp_sc, 1e-9):.2f}')
print('')
print('Interpretation:')
print('  - If MLP Euclid R² >> Ridge Euclid R², Euclid info is present in the bottleneck')
print('    but stored nonlinearly (this is a fine outcome — the Euclid target decoder')
print('    has its own nonlinear capacity to read it out).')
print('  - If both are similarly low (< 0.2 mean), the bottleneck genuinely under-represents')
print('    Euclid spectral content. v9 adversarial masking or architectural changes are warranted.')


## 3. Verdict

Combine the two diagnostics into a single decision.  The thresholds are chosen to be deliberately conservative — we want clear signals before recommending a v9 retrain.

In [ ]:
# Verdict — combines Diag 1 (gradient attribution) with Diag 2b (source-centered MLP probe).
# Diag 2 (random + ridge) is kept as a sanity baseline but the verdict uses the more honest
# Diag 2b numbers, since the bottleneck is read out by nonlinear decoders downstream.
rubin_r2_mean   = float(np.mean([r2_per_band[b] for b in RUBIN_BANDS  if b in r2_per_band]))
euclid_r2_mean  = float(np.mean([r2_per_band[b] for b in EUCLID_BANDS if b in r2_per_band]))
euclid_over_rubin_random_ridge = euclid_r2_mean / max(rubin_r2_mean, 1e-9)

rubin_mlp   = float(np.mean([r2_mlp_sc[b]   for b in RUBIN_BANDS  if b in r2_mlp_sc]))
euclid_mlp  = float(np.mean([r2_mlp_sc[b]   for b in EUCLID_BANDS if b in r2_mlp_sc]))
rubin_ridge_sc_mean  = float(np.mean([r2_ridge_sc[b] for b in RUBIN_BANDS  if b in r2_ridge_sc]))
euclid_ridge_sc_mean = float(np.mean([r2_ridge_sc[b] for b in EUCLID_BANDS if b in r2_ridge_sc]))
euclid_over_rubin_mlp = euclid_mlp / max(rubin_mlp, 1e-9)

diag1_pass = mean_cross_over_within > 0.30
diag1_marg = (0.10 < mean_cross_over_within <= 0.30)
# Diag 2 verdict uses MLP probe (more honest) as primary; ridge is a sanity check.
diag2_pass = euclid_over_rubin_mlp > 0.7
diag2_marg = (0.4 < euclid_over_rubin_mlp <= 0.7)

print('=' * 70)
print('Diagnostic 1 — gradient attribution (input-side)')
print(f'  mean cross-instrument fraction / within-instrument fraction = {mean_cross_over_within:.2f}')
print(f'  pass: > 0.30, marginal: 0.10–0.30, fail: < 0.10')
print(f'  result: {"PASS" if diag1_pass else "MARGINAL" if diag1_marg else "FAIL"}')
print('-' * 70)
print('Diagnostic 2 — linear probe (random points + ridge, sanity baseline)')
print(f'  Rubin mean R²  = {rubin_r2_mean:.3f}')
print(f'  Euclid mean R² = {euclid_r2_mean:.3f}')
print(f'  Euclid / Rubin = {euclid_over_rubin_random_ridge:.2f}')
print('-' * 70)
print('Diagnostic 2b — source-centered probe (the verdict input)')
print(f'  Ridge:  Rubin={rubin_ridge_sc_mean:.3f}  Euclid={euclid_ridge_sc_mean:.3f}  '
      f'ratio={euclid_ridge_sc_mean/max(rubin_ridge_sc_mean, 1e-9):.2f}')
print(f'  MLP:    Rubin={rubin_mlp:.3f}  Euclid={euclid_mlp:.3f}  '
      f'ratio={euclid_over_rubin_mlp:.2f}   <-- verdict uses this')
print(f'  pass: > 0.70, marginal: 0.40–0.70, fail: < 0.40')
print(f'  result: {"PASS" if diag2_pass else "MARGINAL" if diag2_marg else "FAIL"}')
print('=' * 70)

# Also report "is Euclid info nonlinearly stored?" — MLP gain over ridge on source-centered data.
euclid_mlp_gain = euclid_mlp - euclid_ridge_sc_mean
rubin_mlp_gain  = rubin_mlp  - rubin_ridge_sc_mean
print(f'\nMLP–Ridge gap on source-centered data:')
print(f'  Rubin gain (MLP − Ridge)  = {rubin_mlp_gain:+.3f}')
print(f'  Euclid gain (MLP − Ridge) = {euclid_mlp_gain:+.3f}')
if euclid_mlp_gain > 0.10 and euclid_mlp_gain > rubin_mlp_gain:
    print('  → Euclid info is stored nonlinearly in the bottleneck (the Euclid target decoder')
    print('    can read it out through its own ConvNeXt blocks).')
elif euclid_mlp < 0.20:
    print('  → Euclid info is genuinely under-represented in the bottleneck (linear and')
    print('    nonlinear probes both fail).')
else:
    print('  → Euclid info is partially recoverable; ambiguous result.')

if diag1_pass and diag2_pass:
    verdict = ('Cross-instrument signal is genuinely in the bottleneck — both the input '
               'gradient flow AND the bottleneck content show it. v9 adversarial masking not urgent.')
elif diag1_pass and not diag2_pass:
    verdict = ('Encoder pulls from cross-instrument inputs (likely VIS as a spatial scaffold) '
               'but the bottleneck does not store cross-instrument spectral info even nonlinearly. '
               'The bottleneck is acting as a Rubin-centric SED store with a VIS spatial anchor. '
               'Downstream Euclid heads work because their decoders compensate, but a foundation '
               'change (architectural symmetry — concat fusion for Rubin too — or adversarial masking) '
               'would be a meaningful upgrade.')
elif not diag1_pass and diag2_pass:
    verdict = ('Bottleneck preserves Euclid info, but encoder is not actively using cross-instrument '
               'inputs at reconstruction time. The Euclid stream is informative on its own; '
               'v9 masking would force coupling.')
else:
    verdict = ('Cross-instrument signal is weak in BOTH gradient attribution and bottleneck content. '
               'v9 adversarial masking is justified — the model is currently doing within-instrument '
               'reconstruction with the cross-instrument bands as decoration.')

print('\nVerdict:', verdict)

# Save full sidecar JSON for the doc.
import json
summary = {
    'n_tiles': len(selected),
    'diag1_mean_cross_over_within': mean_cross_over_within,
    'diag1_per_target_within_frac': {ALL_BANDS[i]: float(frac_within[i]) for i in range(n_b)},
    'diag1_per_target_cross_frac':  {ALL_BANDS[i]: float(frac_cross[i])  for i in range(n_b)},
    'diag2_random_ridge_r2_per_band': r2_per_band,
    'diag2_random_ridge_rubin_mean': rubin_r2_mean,
    'diag2_random_ridge_euclid_mean': euclid_r2_mean,
    'diag2b_source_centered_ridge_r2_per_band': r2_ridge_sc,
    'diag2b_source_centered_mlp_r2_per_band':   r2_mlp_sc,
    'diag2b_mlp_rubin_mean':  rubin_mlp,
    'diag2b_mlp_euclid_mean': euclid_mlp,
    'diag2b_mlp_euclid_over_rubin': euclid_over_rubin_mlp,
    'verdict': verdict,
}
with open(OUT / 'cross_instrument_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSummary saved to {OUT / "cross_instrument_summary.json"}')


## 4. What to do next

- **If both diagnostics pass:** the cross-instrument flow is real. Document the result in the foundation-model section of `DOCUMENTATION.md` and continue with v8 as production. No retrain needed.
- **If one fails:** rerun this notebook with more tiles (set `N_TILES = 30` or so) to tighten the statistics. If the result holds, the answer is unambiguous.
- **If both fail:** plan v9 with adversarial masking. The change is conceptually simple — modify `sample_context_target_phaseB_mixed` so that with some probability the masking step removes the target's two within-instrument neighbours together, forcing the encoder to use cross-instrument bands. The architecture stays the same, downstream heads stay the same, only the foundation training script and one checkpoint need to be re-run.

Either way, this notebook is the experimental record of what we found and why we chose the next step.